In [1]:
!pip install pyyaml


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# conversion of yaml files to CSV files

import yaml
import os
import glob
from datetime import datetime
from collections import defaultdict
import pandas as pd

# Configuration
INPUT_ROOT = 'D:/python_vs/stocks_analysis/data'
OUTPUT_DIR = 'output_csvs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def process_yaml_file(file_path):
  
    try:
        with open(file_path, 'r') as f:
            data = yaml.safe_load(f)
        
        if not data:
            return []
            
        records = data if isinstance(data, list) else [data]
        processed = []
        
        for record in records:
            if not isinstance(record, dict):
                continue
                
            # Handle case-sensitive field names
            ticker = record.get('Ticker') or record.get('ticker')
            if not ticker:
                continue
                
            # Standardize the record format
            standardized = {
                'ticker': ticker,
                'date': record.get('Date') or record.get('date'),
                'open': record.get('Open') or record.get('open'),
                'high': record.get('High') or record.get('high'),
                'low': record.get('Low') or record.get('low'),
                'close': record.get('Close') or record.get('close'),
                'volume': record.get('Volume') or record.get('volume')
            }
            
            # Add filename timestamp if available
            try:
                dt = datetime.strptime(os.path.basename(file_path)[:19], '%Y-%m-%d_%H-%M-%S')
                standardized['timestamp'] = dt.strftime('%Y-%m-%d %H:%M:%S')
                if not standardized['date']:
                    standardized['date'] = dt.strftime('%Y-%m-%d')
            except:
                pass
                
            processed.append(standardized)
            
        return processed
        
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return []

# Main processing
ticker_data = defaultdict(list)

for yaml_file in glob.glob(os.path.join(INPUT_ROOT, '**/*.yaml'), recursive=True):
    records = process_yaml_file(yaml_file)
    for record in records:
        ticker_data[record['ticker']].append(record)

# Save to CSV
for ticker, records in ticker_data.items():
    df = pd.DataFrame(records)
    df.sort_values(by=['date', 'timestamp'], inplace=True)
    df.to_csv(os.path.join(OUTPUT_DIR, f"{ticker}.csv"), index=False)

print(f"Processed {len(ticker_data)} tickers")

Processed 50 tickers


In [ ]:
#Volatility Analysis

import pandas as pd
import numpy as np
import os

# Configuration
INPUT_DIR = 'output_csvs'  # Directory containing your ticker CSV files
VOLATILITY_RESULTS = 'volatility_analysis.csv'  # Output file for results

def calculate_volatility(ticker_files):
    """Calculate daily returns and volatility for each stock"""
    results = []
    
    for file in ticker_files:
        try:
            df = pd.read_csv(file)
            
            # Ensure data is sorted by date
            df['date'] = pd.to_datetime(df['date'])
            df = df.sort_values('date')
            
            # Calculate daily returns
            df['daily_return'] = df['close'].pct_change()
            
            # Calculate volatility (std dev of daily returns)
            volatility = df['daily_return'].std() * np.sqrt(284)  # Annualized volatility
            
            # Store results
            ticker = os.path.splitext(os.path.basename(file))[0]
            results.append({
                'Ticker': ticker,
                'Volatility': volatility,
                })
            
                    
        except Exception as e:
            print(f"Error processing {file}: {str(e)}")
    
    return pd.DataFrame(results)

# Get all ticker CSV files
ticker_files = [os.path.join(INPUT_DIR, f) for f in os.listdir(INPUT_DIR) 
               if f.endswith('.csv') and not f.endswith('_with_returns.csv')]

# Calculate volatility for all stocks
volatility_df = calculate_volatility(ticker_files)

# Save results
volatility_df.to_csv(VOLATILITY_RESULTS, index=False)

print(f"\nResults saved to {VOLATILITY_RESULTS}")

Top 10 Most Volatile Stocks:
        Ticker  Volatility
0     ADANIENT    0.481985
1   ADANIPORTS    0.438642
8          BEL    0.392379
47       TRENT    0.388847
34        ONGC    0.374918
10        BPCL    0.371911
39  SHRIRAMFIN    0.365470
13   COALINDIA    0.360832
21    HINDALCO    0.330083
33        NTPC    0.328199

Volatility Analysis Summary:
Most Volatile Stock: ADANIENT (0.4820)
Least Volatile Stock: SUNPHARMA (0.1977)

Results saved to volatility_analysis.csv


In [ ]:
import pandas as pd

dia=pd.read_csv("D:/python_vs/stocks_analysis/volatility_analysis.csv")

display(dia)

,Ticker,Volatility
0,ADANIENT,0.481985
1,ADANIPORTS,0.438642
2,APOLLOHOSP,0.238200
3,ASIANPAINT,0.213429
4,AXISBANK,0.263317
5,BAJAJ-AUTO,0.296758
6,BAJAJFINSV,0.237918
7,BAJFINANCE,0.268263
8,BEL,0.392379
9,BHARTIARTL,0.229604


In [ ]:
#cumulative returns ticker wise

import pandas as pd
import numpy as np
import os

# Configuration
INPUT_DIR = 'output_csvs'  # Folder containing individual ticker CSV files
OUTPUT_DIR = 'analysis_results'  # Output folder
os.makedirs(OUTPUT_DIR, exist_ok=True)

def analyze_all_stocks():
    # Initialize DataFrames to store all results
    all_cumulative = pd.DataFrame()
   
    # Get all ticker files
    ticker_files = [f for f in os.listdir(INPUT_DIR) if f.endswith('.csv')]
    
    for file in ticker_files:
        try:
            ticker = os.path.splitext(file)[0]
            file_path = os.path.join(INPUT_DIR, file)
            
            # Read and prepare data
            df = pd.read_csv(file_path)
            df['date'] = pd.to_datetime(df['date'])
            df = df.sort_values('date')
            
            # Calculate returns
            df['daily_return'] = df['close'].pct_change()
            df['cumulative_return'] = (1 + df['daily_return']).cumprod() - 1
            
            # Calculate volatility metrics
            daily_vol = df['daily_return'].std()
            annual_vol = daily_vol * np.sqrt(252)
            
            # Add ticker column
            df['ticker'] = ticker
            
            # Append to cumulative returns DataFrame
            cumulative_data = df[['ticker', 'date', 'close', 'cumulative_return']]
            all_cumulative = pd.concat([all_cumulative, cumulative_data])
            
            
        except Exception as e:
            print(f"Error processing {file}: {str(e)}")
    
    # Save consolidated files
    cumulative_file = os.path.join(OUTPUT_DIR, 'all_cumulative_returns.csv')
       
    all_cumulative.to_csv(cumulative_file, index=False)
  
    
    print(f"\nAnalysis complete! Results saved to {OUTPUT_DIR}")
    print(f"-> Cumulative returns: {cumulative_file}")
   
    
    return all_cumulative

# Run the analysis
cumulative_df = analyze_all_stocks()

# Display sample results
print("\nSample Cumulative Returns:")
print(cumulative_df.head())



Analysis complete! Results saved to analysis_results
-> Cumulative returns: analysis_results\all_cumulative_returns.csv
-> Volatility analysis: analysis_results\all_volatility_analysis.csv

Sample Cumulative Returns:
     ticker                date    close  cumulative_return
0  ADANIENT 2023-10-03 05:30:00  2387.25                NaN
1  ADANIENT 2023-10-04 05:30:00  2464.95           0.032548
2  ADANIENT 2023-10-05 05:30:00  2466.35           0.033134
3  ADANIENT 2023-10-06 05:30:00  2478.10           0.038056
4  ADANIENT 2023-10-09 05:30:00  2442.60           0.023186

Sample Volatility Analysis:
       ticker  annualized_volatility
0    ADANIENT               0.454019
0  ADANIPORTS               0.413192
0  APOLLOHOSP               0.224379
0  ASIANPAINT               0.201046
0    AXISBANK               0.248039


In [ ]:
#284 days return date wise  for calculating sector peformance


import pandas as pd
import os

# Configuration
INPUT_DIR = 'output_csvs'  # Folder containing stock CSV files
OUTPUT_FILE = '284day_returns.csv'  # Results file
TOTAL_DAYS = 284  # Your specific data period length

def calculate_284day_returns():
    results = []
    
    for filename in os.listdir(INPUT_DIR):
        if filename.endswith('.csv'):
            try:
                # Load data
                filepath = os.path.join(INPUT_DIR, filename)
                df = pd.read_csv(filepath)
                df['date'] = pd.to_datetime(df['date'])
                df = df.sort_values('date')
                
                if len(df) < TOTAL_DAYS:
                    print(f"Skipping {filename}: Only {len(df)} days of data")
                    continue
                
                # Get first and last prices (284 trading days apart)
                initial_price = df['close'].iloc[0]
                final_price = df['close'].iloc[TOTAL_DAYS-1]  # 284th day (0-indexed)
                
                # Calculate returns
                total_return = (final_price - initial_price) / initial_price
                
                
                results.append({
                    'ticker': os.path.splitext(filename)[0],
                    'start_date': df['date'].iloc[0].strftime('%Y-%m-%d'),
                    'end_date': df['date'].iloc[TOTAL_DAYS-1].strftime('%Y-%m-%d'),
                    'days_held': TOTAL_DAYS,
                    'initial_price': initial_price,
                    'final_price': final_price,
                    'total_return': total_return,
                    'total_return_pct': f"{total_return*100:.2f}%",
                    
                })
                
            except Exception as e:
                print(f"Error processing {filename}: {e}")
    
    # Save results
    result_df = pd.DataFrame(results)
    result_df.to_csv(OUTPUT_FILE, index=False)
    print(f"\nReturns over {TOTAL_DAYS} trading days saved to {OUTPUT_FILE}")
    print(f"Analyzed {len(results)} stocks with complete data")
    return result_df

# Execute
returns_284day = calculate_284day_returns()

# Display sample results
print("\nSample Results:")
print(returns_284day[['ticker', 'start_date', 'end_date', 'total_return_pct']].head())


Returns over 284 trading days saved to 284day_returns.csv
Analyzed 50 stocks with complete data

Sample Results:
       ticker  start_date    end_date total_return_pct
0    ADANIENT  2023-10-03  2024-11-22           -6.67%
1  ADANIPORTS  2023-10-03  2024-11-22           36.73%
2  APOLLOHOSP  2023-10-03  2024-11-22           35.48%
3  ASIANPAINT  2023-10-03  2024-11-22          -21.94%
4    AXISBANK  2023-10-03  2024-11-22            9.74%


In [ ]:
#merging sector and return data for sector performance

import pandas as pd

# Load your data
sector_data = pd.read_csv('Sector_data.csv')  
returns_data = pd.read_csv('284day_returns.csv')  

# Merge the DataFrames on ticker
merged_data = pd.merge(
    returns_data,
    sector_data,
    on='ticker',
    how='left'  # Keeps all stocks from returns_data even if sector is missing
)

# Save merged data
merged_data.to_csv('sector_returns_284day.csv', index=False)
print("Merged data saved to 'sector_returns_284day.csv'")



Merged data saved to 'sector_returns_284day.csv'


In [ ]:
#mothly return by ticker

import pandas as pd
import os

# Configuration
INPUT_DIR = 'output_csvs'  # Directory containing stock CSV files
OUTPUT_DIR = 'monthly_returns'  # Output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

def calculate_monthly_returns():
    """Calculate monthly returns for each stock"""
    all_results = []
    
    for filename in os.listdir(INPUT_DIR):
        if filename.endswith('.csv'):
            try:
                ticker = os.path.splitext(filename)[0]
                df = pd.read_csv(os.path.join(INPUT_DIR, filename))
                
                # Convert and sort dates
                df['date'] = pd.to_datetime(df['date'])
                df = df.sort_values('date')
                
                # Extract year-month
                df['year_month'] = df['date'].dt.to_period('M')
                
                # Group by month
                monthly_data = df.groupby('year_month').agg({
                    'date': ['first', 'last'],
                    'close': ['first', 'last']
                })
                
                # Flatten multi-index columns
                monthly_data.columns = ['_'.join(col).strip() for col in monthly_data.columns.values]
                monthly_data = monthly_data.rename(columns={
                    'date_first': 'start_date',
                    'date_last': 'end_date',
                    'close_first': 'start_price',
                    'close_last': 'end_price'
                })
                
                # Calculate monthly returns
                monthly_data['monthly_return'] = (monthly_data['end_price'] - monthly_data['start_price']) / monthly_data['start_price']
                monthly_data['ticker'] = ticker
                
                # Save individual stock results
                output_file = os.path.join(OUTPUT_DIR, f"{ticker}_monthly_returns.csv")
                monthly_data.to_csv(output_file)
                
                all_results.append(monthly_data)
                
            except Exception as e:
                print(f"Error processing {filename}: {e}")
    
    # Combine all results
    combined_results = pd.concat(all_results)
    combined_file = os.path.join(OUTPUT_DIR, 'all_stocks_monthly_returns.csv')
    combined_results.to_csv(combined_file)
    
    print(f"\nMonthly returns calculated for {len(all_results)} stocks")
    print(f"Individual files saved to {OUTPUT_DIR}")
    print(f"Combined results saved to {combined_file}")
    
    return combined_results

# Execute
monthly_returns = calculate_monthly_returns()

# Display sample results
print("\nSample Monthly Returns:")
print(monthly_returns.head())


Monthly returns calculated for 50 stocks
Individual files saved to monthly_returns
Combined results saved to monthly_returns\all_stocks_monthly_returns.csv

Sample Monthly Returns:
                    start_date            end_date  start_price  end_price  \
year_month                                                                   
2023-10    2023-10-03 05:30:00 2023-10-31 05:30:00      2387.25    2294.65   
2023-11    2023-11-01 05:30:00 2023-11-30 05:30:00      2217.30    2358.55   
2023-12    2023-12-01 05:30:00 2023-12-29 05:30:00      2362.70    2848.95   
2024-01    2024-01-01 05:30:00 2024-01-31 05:30:00      2917.20    3142.00   
2024-02    2024-02-01 05:30:00 2024-02-29 05:30:00      3153.50    3285.40   

            monthly_return    ticker  
year_month                            
2023-10          -0.038789  ADANIENT  
2023-11           0.063704  ADANIENT  
2023-12           0.205803  ADANIENT  
2024-01           0.077060  ADANIENT  
2024-02           0.041827  ADANIENT 

Data Storage Process

In [ ]:
import mysql.connector

connection = mysql.connector.connect(
  host = "gateway01.ap-southeast-1.prod.aws.tidbcloud.com",
  port = 4000,
  user = "kCCeTyfqG4q97x6.root",
  password = "O5K4JarXblpcn7gg",
  database = "stock_analysis",

)

mycursor = connection.cursor(buffered=True)

In [ ]:
mycursor.execute("create database stock_analysis")

In [ ]:
mycursor.execute("create table stock_analysis.volatility (Ticker varchar(255) PRIMARY KEY, Volatility FLOAT NOT NULL) ")

#Ticker,Volatility

In [ ]:
import pandas as pd

df=pd.read_csv("volatility_analysis.csv")

display(df)

In [ ]:
insert_query = "INSERT INTO stock_analysis.volatility (Ticker,Volatility) VALUES (%s, %s)"
values = df[['Ticker', 'Volatility']].values.tolist()   

mycursor.executemany(insert_query, values)  
connection.commit()  


In [ ]:
mycursor.execute("CREATE TABLE stock_analysis.monthly_return (Ticker VARCHAR(255), yr_mon DATE NOT NULL, returns_monthly FLOAT NOT NULL)")

In [ ]:
import pandas as pd

df=pd.read_csv("D:/python_vs/stocks_analysis/monthly_returns/all_stocks_monthly_returns.csv")

display(df)

,year_month,start_date,end_date,start_price,end_price,monthly_return,ticker
0,2023-10,2023-10-03 05:30:00,2023-10-31 05:30:00,2387.25,2294.65,-0.038789,ADANIENT
1,2023-11,2023-11-01 05:30:00,2023-11-30 05:30:00,2217.30,2358.55,0.063704,ADANIENT
2,2023-12,2023-12-01 05:30:00,2023-12-29 05:30:00,2362.70,2848.95,0.205803,ADANIENT
3,2024-01,2024-01-01 05:30:00,2024-01-31 05:30:00,2917.20,3142.00,0.077060,ADANIENT
4,2024-02,2024-02-01 05:30:00,2024-02-29 05:30:00,3153.50,3285.40,0.041827,ADANIENT
...,...,...,...,...,...,...,...
695,2024-07,2024-07-01 05:30:00,2024-07-31 05:30:00,527.35,522.00,-0.010145,WIPRO
696,2024-08,2024-08-01 05:30:00,2024-08-30 05:30:00,521.55,538.40,0.032308,WIPRO
697,2024-09,2024-09-02 05:30:00,2024-09-30 05:30:00,532.45,541.45,0.016903,WIPRO
698,2024-10,2024-10-01 05:30:00,2024-10-31 05:30:00,546.75,551.80,0.009236,WIPRO


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 700 entries, 0 to 699
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   year_month      700 non-null    datetime64[ns]
 1   start_date      700 non-null    object        
 2   end_date        700 non-null    object        
 3   start_price     700 non-null    float64       
 4   end_price       700 non-null    float64       
 5   monthly_return  700 non-null    float64       
 6   ticker          700 non-null    object        
dtypes: datetime64[ns](1), float64(3), object(3)
memory usage: 38.4+ KB


In [ ]:
df['year_month'] = pd.to_datetime(df['year_month'], format='%Y-%m')


In [ ]:
insert_query = "INSERT INTO stock_analysis.monthly_return (Ticker,yr_mon,returns_monthly) VALUES (%s, %s,%s)"
values = df[['ticker', 'year_month','monthly_return']].values.tolist()   

mycursor.executemany(insert_query, values)  
connection.commit()  


In [ ]:
import pandas as pd

df=pd.read_csv("D:/python_vs/stocks_analysis/analysis_results/all_cumulative_returns.csv")

df

,ticker,date,close,cumulative_return
0,ADANIENT,2023-10-03 05:30:00,2387.25,NaN
1,ADANIENT,2023-10-04 05:30:00,2464.95,0.032548
2,ADANIENT,2023-10-05 05:30:00,2466.35,0.033134
3,ADANIENT,2023-10-06 05:30:00,2478.10,0.038056
4,ADANIENT,2023-10-09 05:30:00,2442.60,0.023186
...,...,...,...,...
14195,WIPRO,2024-11-14 05:30:00,566.70,0.397706
14196,WIPRO,2024-11-18 05:30:00,552.85,0.363547
14197,WIPRO,2024-11-19 05:30:00,562.00,0.386114
14198,WIPRO,2024-11-21 05:30:00,557.15,0.374152


In [ ]:
df['cumulative_return'] = df['cumulative_return'].fillna(0)

In [ ]:
display(df)

,ticker,date,close,cumulative_return
0,ADANIENT,2023-10-03,2387.25,0.000000
1,ADANIENT,2023-10-04,2464.95,0.032548
2,ADANIENT,2023-10-05,2466.35,0.033134
3,ADANIENT,2023-10-06,2478.10,0.038056
4,ADANIENT,2023-10-09,2442.60,0.023186
...,...,...,...,...
14195,WIPRO,2024-11-14,566.70,0.397706
14196,WIPRO,2024-11-18,552.85,0.363547
14197,WIPRO,2024-11-19,562.00,0.386114
14198,WIPRO,2024-11-21,557.15,0.374152


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14200 entries, 0 to 14199
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   ticker             14200 non-null  object        
 1   date               14200 non-null  datetime64[ns]
 2   close              14200 non-null  float64       
 3   cumulative_return  14200 non-null  float64       
dtypes: datetime64[ns](1), float64(2), object(1)
memory usage: 443.9+ KB


In [ ]:
df['date'] = pd.to_datetime(df['date'])

In [ ]:
df.to_csv("cleaned_all_cumulative_returns.csv", index=False)

In [ ]:
display(df)

,ticker,date,close,cumulative_return
0,ADANIENT,2023-10-03,2387.25,0.000000
1,ADANIENT,2023-10-04,2464.95,0.032548
2,ADANIENT,2023-10-05,2466.35,0.033134
3,ADANIENT,2023-10-06,2478.10,0.038056
4,ADANIENT,2023-10-09,2442.60,0.023186
...,...,...,...,...
14195,WIPRO,2024-11-14,566.70,0.397706
14196,WIPRO,2024-11-18,552.85,0.363547
14197,WIPRO,2024-11-19,562.00,0.386114
14198,WIPRO,2024-11-21,557.15,0.374152


In [ ]:
mycursor.execute("CREATE TABLE stock_analysis.cumulative_return (Ticker VARCHAR(255), tran_date DATE NOT NULL,close_price FLOAT NOT NULL, cumulative_return FLOAT NOT NULL)")

In [ ]:
insert_query = "INSERT INTO stock_analysis.cumulative_return (Ticker,tran_date,close_price,cumulative_return) VALUES (%s, %s,%s,%s)"
values = df[['ticker', 'date','close','cumulative_return']].values.tolist()   

mycursor.executemany(insert_query, values)  
connection.commit()  

In [ ]:
import pandas as pd
ad=pd.read_csv("sector_returns_284day.csv")
ad

,ticker,start_date,end_date,days_held,initial_price,final_price,total_return,total_return_pct,COMPANY,sector,Symbol
0,ADANIENT,03-10-2023,22-11-2024,284,2387.25,2228.00,-0.066709,-6.67%,NaN,NaN,NaN
1,ADANIPORTS,03-10-2023,22-11-2024,284,831.40,1136.75,0.367272,36.73%,ADANI PORTS & SEZ,MISCELLANEOUS,ADANI PORTS & SEZ
2,APOLLOHOSP,03-10-2023,22-11-2024,284,5118.95,6935.10,0.354790,35.48%,APOLLO HOSPITALS,MISCELLANEOUS,APOLLO HOSPITALS
3,ASIANPAINT,03-10-2023,22-11-2024,284,3166.85,2472.20,-0.219350,-21.94%,ASIAN PAINTS,PAINTS,ASIAN PAINTS
4,AXISBANK,03-10-2023,22-11-2024,284,1041.05,1142.40,0.097354,9.74%,AXIS BANK,BANKING,AXIS BANK
5,BAJAJ-AUTO,03-10-2023,22-11-2024,284,5016.45,9481.65,0.890112,89.01%,BAJAJ AUTO,AUTOMOBILES,BAJAJ AUTO
6,BAJAJFINSV,03-10-2023,22-11-2024,284,1561.05,1600.85,0.025496,2.55%,BAJAJ FINSERV,FINANCE,BAJAJ FINSERV
7,BAJFINANCE,03-10-2023,22-11-2024,284,7967.60,6683.95,-0.161109,-16.11%,BAJAJ FINANCE,FINANCE,BAJAJ FINANCE
8,BEL,03-10-2023,22-11-2024,284,139.20,280.85,1.017601,101.76%,BHARAT ELECTRONICS,DEFENCE,BHARAT ELECTRONICS
9,BHARTIARTL,03-10-2023,22-11-2024,284,925.30,1569.30,0.695990,69.60%,NaN,NaN,NaN


In [ ]:
cleaned_ad=ad.dropna()
display(cleaned_ad)

,ticker,start_date,end_date,days_held,initial_price,final_price,total_return,total_return_pct,COMPANY,sector,Symbol
1,ADANIPORTS,03-10-2023,22-11-2024,284,831.40,1136.75,0.367272,36.73%,ADANI PORTS & SEZ,MISCELLANEOUS,ADANI PORTS & SEZ
2,APOLLOHOSP,03-10-2023,22-11-2024,284,5118.95,6935.10,0.354790,35.48%,APOLLO HOSPITALS,MISCELLANEOUS,APOLLO HOSPITALS
3,ASIANPAINT,03-10-2023,22-11-2024,284,3166.85,2472.20,-0.219350,-21.94%,ASIAN PAINTS,PAINTS,ASIAN PAINTS
4,AXISBANK,03-10-2023,22-11-2024,284,1041.05,1142.40,0.097354,9.74%,AXIS BANK,BANKING,AXIS BANK
5,BAJAJ-AUTO,03-10-2023,22-11-2024,284,5016.45,9481.65,0.890112,89.01%,BAJAJ AUTO,AUTOMOBILES,BAJAJ AUTO
6,BAJAJFINSV,03-10-2023,22-11-2024,284,1561.05,1600.85,0.025496,2.55%,BAJAJ FINSERV,FINANCE,BAJAJ FINSERV
7,BAJFINANCE,03-10-2023,22-11-2024,284,7967.60,6683.95,-0.161109,-16.11%,BAJAJ FINANCE,FINANCE,BAJAJ FINANCE
8,BEL,03-10-2023,22-11-2024,284,139.20,280.85,1.017601,101.76%,BHARAT ELECTRONICS,DEFENCE,BHARAT ELECTRONICS
10,BPCL,03-10-2023,22-11-2024,284,170.68,285.85,0.674772,67.48%,BPCL,ENERGY,BPCL
12,CIPLA,03-10-2023,22-11-2024,284,1182.80,1486.50,0.256764,25.68%,CIPLA,PHARMACEUTICALS,CIPLA


In [ ]:
cleaned_ad["total_return_pct"]=cleaned_ad["total_return_pct"].str.replace("%","")

C:\Users\mutha\AppData\Local\Temp\ipykernel_3812\4276659059.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_ad["total_return_pct"]=cleaned_ad["total_return_pct"].str.replace("%","")


In [ ]:
cleaned_ad

,ticker,start_date,end_date,days_held,initial_price,final_price,total_return,total_return_pct,COMPANY,sector,Symbol
1,ADANIPORTS,03-10-2023,22-11-2024,284,831.40,1136.75,0.367272,36.73,ADANI PORTS & SEZ,MISCELLANEOUS,ADANI PORTS & SEZ
2,APOLLOHOSP,03-10-2023,22-11-2024,284,5118.95,6935.10,0.354790,35.48,APOLLO HOSPITALS,MISCELLANEOUS,APOLLO HOSPITALS
3,ASIANPAINT,03-10-2023,22-11-2024,284,3166.85,2472.20,-0.219350,-21.94,ASIAN PAINTS,PAINTS,ASIAN PAINTS
4,AXISBANK,03-10-2023,22-11-2024,284,1041.05,1142.40,0.097354,9.74,AXIS BANK,BANKING,AXIS BANK
5,BAJAJ-AUTO,03-10-2023,22-11-2024,284,5016.45,9481.65,0.890112,89.01,BAJAJ AUTO,AUTOMOBILES,BAJAJ AUTO
6,BAJAJFINSV,03-10-2023,22-11-2024,284,1561.05,1600.85,0.025496,2.55,BAJAJ FINSERV,FINANCE,BAJAJ FINSERV
7,BAJFINANCE,03-10-2023,22-11-2024,284,7967.60,6683.95,-0.161109,-16.11,BAJAJ FINANCE,FINANCE,BAJAJ FINANCE
8,BEL,03-10-2023,22-11-2024,284,139.20,280.85,1.017601,101.76,BHARAT ELECTRONICS,DEFENCE,BHARAT ELECTRONICS
10,BPCL,03-10-2023,22-11-2024,284,170.68,285.85,0.674772,67.48,BPCL,ENERGY,BPCL
12,CIPLA,03-10-2023,22-11-2024,284,1182.80,1486.50,0.256764,25.68,CIPLA,PHARMACEUTICALS,CIPLA


In [ ]:
mycursor.execute("""CREATE TABLE 
                 stock_analysis.sector_yearly_return 
                 (Ticker VARCHAR(255),
                 start_date DATE NOT NULL,
                 end_date DATE NOT NULL,
                 initial_price FLOAT NOT NULL,
                 final_price FLOAT NOT NULL,
                 total_return FLOAT NOT NULL,
                 total_return_pct FLOAT NOT NULL,
                 company_name VARCHAR(255),
                 sector VARCHAR(255)
                 )""")

In [29]:
cleaned_ad.info()

<class 'pandas.core.frame.DataFrame'>
Index: 46 entries, 1 to 49
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ticker            46 non-null     object        
 1   start_date        46 non-null     datetime64[ns]
 2   end_date          46 non-null     datetime64[ns]
 3   days_held         46 non-null     int64         
 4   initial_price     46 non-null     float64       
 5   final_price       46 non-null     float64       
 6   total_return      46 non-null     float64       
 7   total_return_pct  46 non-null     object        
 8   COMPANY           46 non-null     object        
 9   sector            46 non-null     object        
 10  Symbol            46 non-null     object        
dtypes: datetime64[ns](2), float64(3), int64(1), object(5)
memory usage: 4.3+ KB


In [ ]:
cleaned_ad["start_date"]=pd.to_datetime(cleaned_ad["start_date"])
cleaned_ad["end_date"]=pd.to_datetime(cleaned_ad["end_date"])

In [30]:
insert_query = """INSERT INTO stock_analysis.sector_yearly_return (
                Ticker,start_date,end_date,initial_price,final_price,total_return,total_return_pct,company_name,sector) 
                VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s)"""
values = cleaned_ad[['ticker', 'start_date','end_date','initial_price','final_price','total_return','total_return_pct','COMPANY','sector']].values.tolist()   

mycursor.executemany(insert_query, values)  
connection.commit()  